# Customer Segmentation & Retention Analysis — E-Commerce
## Phase 1: Exploratory Data Analysis

**Dataset:** UCI Online Retail II — ~1M real transactions from a UK-based online retailer (2009–2011)  
**Goal:** Understand customer behavior, revenue patterns, and product performance  
**Audience:** This notebook is written for a technical data science audience, but findings are framed in business language to communicate directly with stakeholders at an e-commerce company.

---

## Section 1 — Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

PROCESSED_PATH = '../data/processed/cleaned_retail.csv'

print('Libraries loaded.')

In [ ]:
df = pd.read_csv(
    PROCESSED_PATH,
    parse_dates=['InvoiceDate'],
    dtype={'CustomerID': int}
)

print(f'Shape: {df.shape}')
print(f'Date range: {df["InvoiceDate"].min().date()}  →  {df["InvoiceDate"].max().date()}')
df.head()

In [ ]:
df.dtypes

---
## Section 2 — Revenue Analysis

Revenue is the most fundamental business metric. Before building any predictive model, we need to understand:
- **When** does revenue peak and trough?
- **Where** are customers concentrated geographically?

Answering these shapes how we define customer segments and churn windows later.

In [ ]:
# --- Monthly Revenue Trend ---

monthly_revenue = (
    df.assign(YearMonth=df['InvoiceDate'].dt.to_period('M'))
    .groupby('YearMonth', as_index=False)['TotalPrice']
    .sum()
)
monthly_revenue['YearMonth'] = monthly_revenue['YearMonth'].astype(str)

fig = px.line(
    monthly_revenue,
    x='YearMonth',
    y='TotalPrice',
    markers=True,
    title='Monthly Revenue Trend (2009 – 2011)',
    labels={'YearMonth': 'Month', 'TotalPrice': 'Revenue (£)'},
    template='plotly_white'
)
fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_tickformat='£,.0f',
    height=420
)
fig.show()

**Business Insight — Revenue Seasonality:**  
Revenue shows a strong seasonal spike in **Q4 (October–November)** each year, consistent with holiday gifting demand — this retailer sells giftware. This means customer acquisition and retention campaigns must be timed to:
- **Pre-season (September):** Re-engage lapsed customers before the peak
- **Post-peak (December–January):** Run win-back campaigns for one-time holiday buyers

For churn modeling, we should be mindful that a customer going quiet in November–December may simply be seasonal, not truly churned.

In [ ]:
# --- Top 10 Countries by Revenue ---

country_revenue = (
    df.groupby('Country', as_index=False)['TotalPrice']
    .sum()
    .sort_values('TotalPrice', ascending=False)
    .head(10)
)

fig = px.bar(
    country_revenue,
    x='Country',
    y='TotalPrice',
    title='Top 10 Countries by Total Revenue',
    labels={'TotalPrice': 'Revenue (£)', 'Country': 'Country'},
    color='TotalPrice',
    color_continuous_scale='Blues',
    template='plotly_white'
)
fig.update_layout(
    yaxis_tickformat='£,.0f',
    coloraxis_showscale=False,
    height=420
)
fig.show()

**Business Insight — Geographic Concentration:**  
The **United Kingdom dominates revenue** by a very large margin, reflecting the retailer's home market. A small set of European countries (Germany, France, Netherlands, EIRE) make up most of the international revenue.

For segmentation, we can model UK and international customers jointly — but if we were to build country-level retention strategies, the UK would need its own cohort given its volume. For this project, we will focus on **all customers globally** to maximise dataset size.

---
## Section 3 — Customer Behavior

Understanding *how* customers engage with the platform is the foundation of segmentation. Two distributions matter most:
- **Order frequency:** Are most customers one-time buyers, or do they return regularly?
- **Total spend:** Is revenue driven by a small number of high-spenders, or is it broadly distributed?

The answers directly shape how we will define the RFM segments in Phase 2.

In [ ]:
# --- Customer-level aggregation ---

customer_stats = (
    df.groupby('CustomerID', as_index=False)
    .agg(
        num_orders=('Invoice', 'nunique'),
        total_spend=('TotalPrice', 'sum')
    )
)

print(f'Total customers: {len(customer_stats):,}')
customer_stats.describe()

In [ ]:
# --- Distribution of orders per customer ---
# Cap at 50 orders for readability (long right tail)

orders_capped = customer_stats['num_orders'].clip(upper=50)

fig = px.histogram(
    orders_capped,
    nbins=50,
    title='Distribution of Orders per Customer (capped at 50)',
    labels={'value': 'Number of Orders', 'count': 'Number of Customers'},
    template='plotly_white',
    color_discrete_sequence=['#4C72B0']
)
fig.update_layout(height=400, showlegend=False)
fig.show()

one_time = (customer_stats['num_orders'] == 1).sum()
pct = one_time / len(customer_stats) * 100
print(f'One-time buyers: {one_time:,}  ({pct:.1f}% of all customers)')

**Business Insight — Order Frequency:**  
The distribution is **heavily right-skewed**: the majority of customers placed only 1–3 orders over the entire two-year period. One-time buyers typically make up 30–50% of any e-commerce customer base.

This has direct implications for retention strategy:
- One-time buyers are a high-priority win-back segment
- Customers with 5+ orders are candidates for loyalty or VIP programs
- Churn definition must account for this — a customer inactive for 90 days may simply be a low-frequency buyer

In [ ]:
# --- Distribution of total spend per customer ---
# Use log scale to handle extreme right-skew (a few very high-value wholesale buyers)

spend_positive = customer_stats[customer_stats['total_spend'] > 0]['total_spend']

fig = px.histogram(
    np.log1p(spend_positive),
    nbins=60,
    title='Distribution of Total Spend per Customer (log scale)',
    labels={'value': 'log(1 + Total Spend £)', 'count': 'Number of Customers'},
    template='plotly_white',
    color_discrete_sequence=['#DD8452']
)
fig.update_layout(height=400, showlegend=False)
fig.show()

# Spend percentiles
print('Total spend percentiles:')
print(spend_positive.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).apply(lambda x: f'£{x:,.2f}'))

**Business Insight — Customer Spend Distribution:**  
Customer spend follows a **log-normal distribution** — a small number of high-value buyers (likely wholesale or B2B customers) generate a disproportionate share of revenue. This is the classic Pareto / 80-20 pattern seen across retail.

For CLV modeling (Phase 4), this skew means we must:
- Use log-transformation when clustering on spend (avoid high-spenders dominating clusters)
- Treat extreme outliers carefully — they may be wholesale accounts, not typical consumers

---
## Section 4 — Product Analysis

Understanding which products drive the most volume helps contextualise customer behavior. High-velocity SKUs are important because:
- They indicate what repeat buyers return for
- They can be used as features in the churn model (e.g., did this customer ever buy a top-10 product?)
- They reveal the nature of the business (giftware, stationery, etc.)

In [ ]:
# --- Top 20 best-selling products by quantity sold ---

top_products = (
    df.groupby('Description', as_index=False)['Quantity']
    .sum()
    .sort_values('Quantity', ascending=False)
    .head(20)
)

fig = px.bar(
    top_products.sort_values('Quantity'),
    x='Quantity',
    y='Description',
    orientation='h',
    title='Top 20 Best-Selling Products by Units Sold',
    labels={'Quantity': 'Total Units Sold', 'Description': ''},
    color='Quantity',
    color_continuous_scale='Teal',
    template='plotly_white'
)
fig.update_layout(
    height=620,
    coloraxis_showscale=False,
    yaxis_tickfont_size=11
)
fig.show()

**Business Insight — Top Products:**  
The best-selling products by unit volume are **low-price, high-frequency giftware and home accessories** — items like bags, candle holders, and decorative signs. This is consistent with a wholesale/gifting retailer whose customers often place bulk orders.

This tells us that **Quantity per order is a meaningful behavioral signal** — high-quantity single orders are likely wholesale buyers, not consumers. When building the churn and CLV models, we may need to segment or flag these high-volume wholesale customers separately, as their purchase patterns differ structurally from typical retail customers.

---
## Section 5 — Data Quality Notes

### What was removed and why

| Step | Rows Removed | Reason |
|---|---|---|
| Null CustomerID | ~20–25% of raw rows | Guest transactions with no customer identifier — unusable for any customer-level modeling |
| Duplicate rows | Small number | Likely data pipeline artifacts |
| Cancelled invoices (prefix 'C') | ~2–5% of rows | Represent refunds/returns, not real purchase events |
| Quantity ≤ 0 | Small number | Residual return entries not caught by invoice prefix |
| Price ≤ 0 | Small number | Free items, corrections, data entry errors |

### Assumptions made

1. **Customer identity:** We treat `CustomerID` as a reliable, persistent identifier. In practice, a single person could have multiple IDs (e.g., after a password reset), but we have no way to de-duplicate this.
2. **Cancellations excluded:** We exclude invoices starting with 'C' entirely. We do **not** attempt to match cancellations back to original orders, as this would require order-level reconciliation not supported by this dataset.
3. **No currency adjustment:** All values are in GBP. No inflation adjustment is applied across the two-year span.
4. **Wholesale flag:** We do not formally separate wholesale from retail customers. High-spend, high-quantity outliers are retained but will need attention during RFM scaling.

### What the cleaned dataset looks like

After cleaning, the dataset contains:
- **~750K–900K transaction rows** (vs ~1M+ raw)
- **~5,000–5,900 unique customers**
- **Date range:** December 2009 – December 2011
- **New column:** `TotalPrice = Quantity × Price`
- **`CustomerID`** cast to integer for join compatibility

### Why these decisions matter for modeling

- **RFM table (Phase 2):** All three components — Recency, Frequency, Monetary — depend on clean purchase events. Including cancellations or null customers would distort every downstream metric.
- **Churn model (Phase 3):** The churn label is defined by purchase inactivity over a time window. Cancelled orders must be excluded, or a customer who returned a product would look falsely active.
- **CLV model (Phase 4):** The BG/NBD model fits on purchase frequency and inter-purchase time. Noise from returns or duplicates would corrupt these frequency signals.

The cleaning report (printed by `src/data_processing.py`) gives the exact row counts removed at each step and should be included in any stakeholder presentation.

---
## Next Steps

With EDA complete, Phase 2 will:
1. Build the **RFM table** (Recency, Frequency, Monetary) from `cleaned_retail.csv`
2. Apply **K-Means clustering** to group customers into behavioural segments
3. Label segments in business terms: *Champions, Loyal Customers, At Risk, Hibernating, Lost*

These segments will then feed the churn model (Phase 3) and CLV estimation (Phase 4).